# Baseline Replication: BLIP and DETR
**DS 677 · Florence-2 vs. Specialist Models Under Motion Blur** · Vivek Shah and Victor Figueredo

The claim compares Florence-2 with two single-task baselines: **BLIP-base** for captioning and **DETR-ResNet-50** for detection. Before any blur is added, each baseline is run on all 5,000 clean test images and checked against its published score. This notebook also makes sure each baseline is given a fair chance: BLIP's decoding settings and DETR's matched-protocol threshold are tested, and the best setting is used.

### What the two scripts provide
Both scripts mirror Florence-2 scripts: the same options (`--limit`, `--output-dir`, `--blur-type`, `--blur-px`, `--seed`), blur applied with `apply_blur`, and scoring with `src/evaluate.py`. Blur types, levels and seeds are read from `src/config.py`.

| Script | What it does |
|---|---|
| `src/run_blip_captioning.py` | BLIP-base captioning on the Karpathy test split; 3 beams and up to 128 new tokens (the same as Florence-2). Options `--prompt`, `--dtype` and `--max-new-tokens` (defaults: no prompt, float16, 128). Saves `predictions.json` and `metrics.json` (CIDEr, BLEU-4 and every setting used). |
| `src/run_detr_detection.py` | DETR-ResNet-50 detection on COCO val2017, scored two ways: **official** (DETR's own confidence scores) and **matched** (only confident boxes, all scored 1.0, the same as Florence-2, which has no confidence scores). Option `--matched-threshold`. Saves the predictions and `metrics.json`. |

### What this notebook does
| What happens |
|---|
| Install packages and get the team's latest code |
| Download all the data (~7 GB) and check it is complete |
| Quick test of both scripts on 20 images |
| Find BLIP's best settings on 500 images |
| Full clean BLIP run on 5,000 images |
| Full clean DETR run on 5,000 images |
| Find DETR's best matched-protocol threshold |
| Compare all four models with their published scores |
| Save the clean results to the shared results file |

### Files this notebook produces
- New rows in `experiments/results/results.csv`: BLIP and DETR clean results, next to the Florence-2 rows

## Step 1 · Install packages and get the team's code
- Installs the pinned packages. `pycocoevalcap` scores captions; `pycocotools` scores detections.
- Clones the latest repo, which includes both scripts, and lets Python import from `src/`. It first moves to `/content`: deleting the folder the notebook is currently in would make the clone fail.

**Result:** the latest commit is shown. The pip warnings about `gradio` and `diffusers` are expected and harmless.

In [1]:
# Install pinned package versions
!pip install -q transformers==4.49.0 timm einops pycocotools==2.0.11 pycocoevalcap==1.2

# Step out of the repo folder first (deleting the current folder makes git clone fail)
%cd /content

# Remove any old copy of the repo, then download the latest version from GitHub
!rm -rf /content/repo
!git clone -q https://github.com/NJIT-DS677-Deep-Learning/florence2-vs-specialists-motion-blur.git /content/repo

# Move into the repo and show the latest commit
%cd /content/repo
!git log -1 --oneline

# Let Python import modules from src/
import os, sys
sys.path.insert(0, '/content/repo')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
/content
/content/repo
15b689a (HEAD -> main, origin/main, origin/HEAD) Add motion blur module (linear and shake)


## Step 2 · Download all the data
This notebook needs the files listed below (~7 GB):

| File | Used for |
|---|---|
| `val2014.zip` | Captioning images (the Karpathy test images come from COCO 2014 val) |
| `caption_datasets.zip` | `dataset_coco.json`: which images are in the Karpathy test split, and their human captions |
| `val2017.zip` | Detection images |
| `annotations_trainval2017.zip` | Ground-truth boxes for detection |

The downloads run quietly, which hides error messages, so the cell ends by checking every required file exists. If one is missing it stops with a clear message, before any long run starts. (Without this check, a failed annotations download was only discovered after a 10-minute BLIP run.)

**Result:** `All data present`. Downloading takes about 3–8 minutes.

In [2]:
# Create the data folder (outside the repo, so data is never committed) and move into it
!mkdir -p /content/data
%cd /content/data

# Captioning images: COCO 2014 val (the Karpathy test images come from here)
!wget -q --show-progress http://images.cocodataset.org/zips/val2014.zip
# Karpathy split file: the test images and their human captions
!wget -q --show-progress https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip
# Detection images: COCO 2017 val
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip
# Detection ground truth: box annotations for COCO 2017
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# Unpack each zip (-o overwrites without asking if the cell is run again)
!unzip -q -o val2014.zip
!unzip -q -o caption_datasets.zip
!unzip -q -o val2017.zip
!unzip -q -o annotations_trainval2017.zip

# Make sure dataset_coco.json sits directly in /content/data, then delete the zips
!mv $(find /content/data -name dataset_coco.json | head -1) /content/data/ 2>/dev/null
!rm -f *.zip

# Go back to the repo
%cd /content/repo

# Check every required file exists; stop with a clear message if a download failed
required = ["val2014", "val2017", "dataset_coco.json", "annotations/instances_val2017.json"]
missing = [p for p in required if not os.path.exists(f"/content/data/{p}")]
assert not missing, f"Download failed for: {missing}. Rerun this cell."
print("All data present")

/content/data
val2014.zip         100%[===================>]   6.19G  42.6MB/s    in 2m 36s  
caption_datasets.zi 100%[===================>]  35.04M  45.1MB/s    in 0.8s    
val2017.zip         100%[===================>] 777.80M  48.5MB/s    in 16s     
annotations_trainva 100%[===================>] 241.19M  26.5MB/s    in 9.6s    
/content/repo
All data present


## Step 3 · Quick test on 20 images
Runs both scripts as commands, exactly as they will run on Wulver, on 20 images (about a minute), to check everything works before the long runs. The scores on 20 images are not meaningful; the point is that both runs finish.

**Result:** both models load on `cuda`, each run ends with its `metrics.json`, and the BLIP captions are sensible sentences. The `HF_TOKEN`, TensorFlow and DETR "weights not used" warnings are expected and harmless.

In [3]:
import json

# Run each script on 20 images; show only the last lines of each run
!python -m src.run_blip_captioning --limit 20 --output-dir /content/runs/blip_test 2>&1 | tail -n 18
!python -m src.run_detr_detection --limit 20 --output-dir /content/runs/detr_test 2>&1 | tail -n 24

# Show a few BLIP captions
preds = json.load(open("/content/runs/blip_test/predictions.json"))
for image_id, caption in list(preds.items())[:5]:
  print(image_id, "->", caption)

PTBTokenizer tokenized 187 tokens at 1972.17 tokens per second.
Picked up JAVA_TOOL_OPTIONS: -Xlog:disable
PTBTokenizer tokenized 1256 tokens at 15804.37 tokens per second.
{
  "model": "Salesforce/blip-image-captioning-base",
  "split": "COCO Karpathy test",
  "blur_type": "linear",
  "blur_px": 0,
  "seed": 0,
  "prompt": "",
  "num_beams": 3,
  "max_new_tokens": 128,
  "dtype": "float16",
  "n_images": 20,
  "inference_seconds": 9.89,
  "CIDEr": 113.51474741697322,
  "BLEU-4": 32.43975470237558
}
  "split": "COCO val2017",
  "blur_type": "linear",
  "blur_px": 0,
  "seed": 0,
  "n_images": 20,
  "matched_threshold": 0.5,
  "inference_seconds": 4.54,
  "official": {
    "n_detections": 2000,
    "mAP": 53.528818075236,
    "mAP50": 73.77371866073928,
    "AP_small": 24.196777803502258,
    "AP_medium": 65.07181480909405,
    "AP_large": 77.35482859416256
  },
  "matched": {
    "n_detections": 254,
    "mAP": 47.29779364710111,
    "mAP50": 66.19543940274279,
    "AP_small": 19.17755

## Step 4 · Find BLIP's best settings (500 images)
Section B is graded on whether **the baseline was given a fair chance**, so BLIP is tested with four settings on the same 500 test images before the full run:

| Setting | Prompt | Max new tokens | Precision | Why test it |
|---|---|---|---|---|
| `paper_settings` | "a picture of" | 20 | float16 | The BLIP paper's own decoding settings |
| `prompt_128tok` | "a picture of" | 128 | float16 | The paper's prompt with the project's shared length limit |
| `no_prompt` | none | 128 | float16 | No prompt (the scripts' default) |
| `no_prompt_float32` | none | 128 | float32 | Checks that half precision does not lower the score |

**Result:**

| Setting | CIDEr | BLEU-4 | Avg words | Captions over 20 words |
|---|---|---|---|---|
| `paper_settings` | 97.5 | 27.4 | 6.2 | 0 |
| `prompt_128tok` | 97.5 | 27.4 | 6.3 | 2 |
| `no_prompt` | **110.3** | **33.0** | **8.1** | **0** |
| `no_prompt_float32` | 110.4 | 33.1 | 8.1 | 0 |

- **The prompt hurts.** With "a picture of", CIDEr is about **13 points lower**, captions are about 2 words shorter, and some are garbled, for example *"a plate of food with bro chicken, bro chicken, bro bro…"*. The BLIP paper found the prompt helped slightly, but that does not carry over to the Hugging Face version of the model used here.
- **Precision does not matter.** float16 and float32 score the same to within 0.1, so float16 is used: it runs about 3 times faster.
- **The length limit does not matter.** Capping at 20 tokens gives the same 97.5 CIDEr; it only cuts off the 2 garbled captions.
- **Chosen setting:** no prompt, float16, 128 new tokens (the scripts' defaults).

In [4]:
import statistics

# Four BLIP settings on the same 500 test images
settings = {
  "paper_settings":    '--prompt "a picture of" --max-new-tokens 20 --dtype float16',
  "prompt_128tok":     '--prompt "a picture of" --dtype float16',
  "no_prompt":         '--prompt "" --dtype float16',
  "no_prompt_float32": '--prompt "" --dtype float32',
}
for name, flags in settings.items():
  # Hide the progress lines (errors still show)
  !python -m src.run_blip_captioning --limit 500 {flags} --output-dir /content/runs/blip_{name} > /dev/null

# Compare scores and caption lengths
print(f"{'setting':18s} {'prompt':15s} {'max':>4s} {'precision':9s} {'CIDEr':>6s} {'BLEU-4':>6s} {'avg words':>9s} {'over 20 words':>13s}")
for name in settings:
  m = json.load(open(f"/content/runs/blip_{name}/metrics.json"))
  # Number of words in each caption
  words = [len(c.split()) for c in json.load(open(f"/content/runs/blip_{name}/predictions.json")).values()]
  print(f"{name:18s} {m['prompt']!r:15s} {m['max_new_tokens']:4d} {m['dtype']:9s} {m['CIDEr']:6.1f} {m['BLEU-4']:6.1f} "
        f"{statistics.mean(words):9.1f} {sum(w > 20 for w in words):13d}")

# The longest caption with the prompt shows the garbled output
longest = max(json.load(open("/content/runs/blip_prompt_128tok/predictions.json")).values(), key=lambda c: len(c.split()))
print("\nLongest caption with the prompt:", longest[:150], "...")

I0000 00:00:1795640828.460504    3003 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795640831.664696    3003 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Picked up JAVA_TOOL_OPTIONS: -Xlog:disable
PTBTokenizer tokenized 3608 tokens at 11996.54 tokens per second.
Picked up JAVA_TOOL_OPTIONS: -Xlog:disable
PTBTokenizer tokenized 30539 tokens at 57650.04 tokens per second.
I0000 00:00:1795640898.870544    3381 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795640901.467260    3381 cudart_stub.c

## Step 5 · Full clean BLIP run (5,000 images)
Runs BLIP on all 5,000 Karpathy test images with the chosen setting (the scripts' defaults: no prompt, float16, 3 beams, up to 128 new tokens). About 10 minutes on a T4; only the end of the output is shown.

**Result:** **106.2 CIDEr**, 33.7 BLEU-4, with `"prompt": ""` and `"dtype": "float16"` in `metrics.json`. The first full run, with the prompt, scored 95.4, so the chosen setting gains about 11 points.

In [5]:
# Full clean BLIP run with the default (chosen) settings; show only the last lines
!python -m src.run_blip_captioning --output-dir /content/runs/blip_clean 2>&1 | tail -n 20

Captioned 5000/5000 images
Picked up JAVA_TOOL_OPTIONS: -Xlog:disable
PTBTokenizer tokenized 46722 tokens at 77909.30 tokens per second.
Picked up JAVA_TOOL_OPTIONS: -Xlog:disable
PTBTokenizer tokenized 307093 tokens at 170404.39 tokens per second.
{
  "model": "Salesforce/blip-image-captioning-base",
  "split": "COCO Karpathy test",
  "blur_type": "linear",
  "blur_px": 0,
  "seed": 0,
  "prompt": "",
  "num_beams": 3,
  "max_new_tokens": 128,
  "dtype": "float16",
  "n_images": 5000,
  "inference_seconds": 692.26,
  "CIDEr": 106.1628997883276,
  "BLEU-4": 33.73348988970214
}


## Step 6 · Full clean DETR run (5,000 images)
Runs DETR on all 5,000 val2017 images and scores both protocols. About 15 minutes for the images plus about 2 minutes of scoring; only the end of the output is shown (the two blocks of *Average Precision* lines are the official and matched scores).

**Result:** official **42.07 mAP**, matching DETR's published 42.0 almost exactly. The matched score in `metrics.json` (26.3) uses the default threshold of 0.5; Step 7 finds a fairer threshold.

In [6]:
# Full clean DETR run; show only the last lines (both score blocks and metrics.json)
!python -m src.run_detr_detection --output-dir /content/runs/detr_clean 2>&1 | tail -n 55

 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.460
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.611
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.334
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.533
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.575
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.321
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.629
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.805
Loading and preparing results...
DONE (t=0.26s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=23.00s).
Accumulating evaluation results...
DONE (t=4.37s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.263
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=1

## Step 7 · Find DETR's best matched-protocol threshold
Florence-2 outputs boxes without confidence scores, so it is scored with every box at 1.0. For a fair comparison, DETR is also scored that way: its boxes above a confidence **threshold** are kept and all given 1.0. The threshold of 0.5 was a guess, so a range is tested using the predictions DETR already saved (the model does not run again), and the best is kept.

- Choosing the threshold on the test images **favours DETR**, the baseline, which makes the claim harder to win: the defensible direction.
- The same threshold is used at **every blur level**. Under blur DETR's confidence drops, so more boxes fall below it; that is part of how DETR degrades.

**Result:**

| Threshold | 0.3 | 0.5 | 0.7 | 0.8 | 0.9 | **0.925** | 0.95 | 0.97 | 0.98 | 0.99 |
|---|---|---|---|---|---|---|---|---|---|---|
| Boxes kept | 74,410 | 57,591 | 44,266 | 36,965 | 27,838 | **25,098** | 21,953 | 18,645 | 16,282 | 12,707 |
| Matched mAP | 23.4 | 26.3 | 28.6 | 29.5 | 30.5 | **30.5** | 30.0 | 28.8 | 27.5 | 24.7 |

- **Best threshold: 0.925**, with matched mAP **30.5**, up from 26.3 at the guessed 0.5. It is used for DETR's matched protocol at every blur level.
- The curve rises, peaks and falls: at low thresholds many wrong boxes are kept (low precision); above 0.95 too many correct boxes are dropped (low recall).
- Even at its best, DETR's matched score is **11.6 points below** its official 42.1. Losing the confidence ranking costs DETR a lot, which is exactly why Florence-2 must be compared with DETR's matched score, not its official one.

In [7]:
import contextlib, io
from src.data import load_val2017
from src.evaluate import score_detections

# Ground truth, and DETR's saved predictions with its own confidence scores
detection, coco = load_val2017()
image_ids = [x["image_id"] for x in detection]
official = json.load(open("/content/runs/detr_clean/official_predictions.json"))

def matched_scores(threshold):
  # Matched protocol: keep predictions at or above the threshold, all with score 1.0.
  kept = [{**d, "score": 1.0} for d in official if d["score"] >= threshold]
  # Hide pycocotools long printout
  with contextlib.redirect_stdout(io.StringIO()):
    return {"n_detections": len(kept), **score_detections(kept, coco, image_ids)}

# Score a range of thresholds
thresholds = (0.3, 0.5, 0.7, 0.8, 0.9, 0.925, 0.95, 0.97, 0.98, 0.99)
sweep = {t: matched_scores(t) for t in thresholds}
for t, m in sweep.items():
  print(f"threshold {t:.3f}: {m['n_detections']:6d} boxes  matched mAP {m['mAP']:5.1f}")

# Keep the threshold with the highest matched mAP
BEST_T = max(sweep, key=lambda t: sweep[t]["mAP"])
print(f"\nBest threshold: {BEST_T} (matched mAP {sweep[BEST_T]['mAP']:.1f})")
if BEST_T == thresholds[-1]:
  print("The best is the highest value tested: try higher thresholds")

loading annotations into memory...
Done (t=0.92s)
creating index...
index created!
threshold 0.300:  74410 boxes  matched mAP  23.4
threshold 0.500:  57591 boxes  matched mAP  26.3
threshold 0.700:  44266 boxes  matched mAP  28.6
threshold 0.800:  36965 boxes  matched mAP  29.5
threshold 0.900:  27838 boxes  matched mAP  30.5
threshold 0.925:  25098 boxes  matched mAP  30.5
threshold 0.950:  21953 boxes  matched mAP  30.0
threshold 0.970:  18645 boxes  matched mAP  28.8
threshold 0.980:  16282 boxes  matched mAP  27.5
threshold 0.990:  12707 boxes  matched mAP  24.7

Best threshold: 0.925 (matched mAP 30.5)


## Step 8 · Compare with the published scores
Puts all four models' clean scores next to their published numbers (the replication anchors for Section C), then shows the clean starting point of each head-to-head comparison in the claim. Florence-2's numbers come from its rows in `results.csv`.

**Result:**

| | Published | Ours | Gap |
|---|---|---|---|
| BLIP-base captioning, CIDEr | 133.3 | 106.2 | −27.1 |
| Florence-2 captioning, CIDEr | 140.0 | 140.0 | 0.0 |
| DETR-R50 detection, official mAP | 42.0 | 42.1 | +0.1 |
| DETR-R50 detection, matched mAP (threshold 0.925) | — | 30.5 | — |
| Florence-2 detection, constant-score mAP | 41.4 | 39.9 | −1.5 |

- **Florence-2 and DETR reproduce their published scores,** which shows the data, scoring code and model loading are correct.
- **BLIP stays about 27 CIDEr below its paper** even with its best setting. The same scoring code reproduces Florence-2 exactly, so the gap comes from the Hugging Face BLIP checkpoint and implementation, not the scorer (the prompt result in Step 4 already shows this version behaves differently from the authors' code).

**Clean head-to-head** (the starting point of each comparison in the claim):

| Task | Florence-2 | Baseline | Florence-2 lead |
|---|---|---|---|
| Captioning (CIDEr) | 140.0 | BLIP 106.2 | +33.8 |
| Detection (mAP, no confidence scores) | 39.9 | DETR 30.5 | +9.4 |

Florence-2 starts ahead on both tasks. The claim is about the **relative drop** under blur, (clean − blurred) ÷ clean, which compares each model with its own clean score, so these different starting points do not decide the result by themselves.

In [8]:
import pandas as pd

# Clean results from the two full runs; DETR's matched scores use the best threshold from Step 7
blip = json.load(open("/content/runs/blip_clean/metrics.json"))
detr = json.load(open("/content/runs/detr_clean/metrics.json"))
detr_matched = sweep[BEST_T]

# Florence-2 clean results, from the shared results file
results = pd.read_csv("experiments/results/results.csv")
florence = results[(results.model == "Florence-2-base-ft") & (results.blur_px == 0)].set_index("metric").score

# (what, published score, our score)
rows = [
  ("BLIP-base captioning, CIDEr", 133.3, blip["CIDEr"]),
  ("Florence-2 captioning, CIDEr", 140.0, florence["CIDEr"]),
  ("DETR-R50 detection, official mAP", 42.0, detr["official"]["mAP"]),
  (f"DETR-R50 detection, matched mAP (t={BEST_T})", None, detr_matched["mAP"]),
  ("Florence-2 detection, constant-score mAP", 41.4, florence["mAP"]),
]
print(f"{'':44s} {'published':>9s} {'ours':>7s} {'gap':>7s}")
for name, published, ours in rows:
  # Gap only where a published score exists
  gap = f"{ours - published:+.1f}" if published else "-"
  print(f"{name:44s} {published if published else '-':>9} {ours:7.1f} {gap:>7s}")

# The clean starting point of each comparison in the claim
print("\nClean head-to-head:")
print(f"  Captioning: Florence-2 {florence['CIDEr']:.1f} vs BLIP {blip['CIDEr']:.1f} CIDEr")
print(f"  Detection:  Florence-2 {florence['mAP']:.1f} vs DETR {detr_matched['mAP']:.1f} mAP (both without confidence scores)")

                                             published    ours     gap
BLIP-base captioning, CIDEr                      133.3   106.2   -27.1
Florence-2 captioning, CIDEr                     140.0   140.0    +0.0
DETR-R50 detection, official mAP                  42.0    42.1    +0.1
DETR-R50 detection, matched mAP (t=0.925)            -    30.5       -
Florence-2 detection, constant-score mAP          41.4    39.9    -1.5

Clean head-to-head:
  Captioning: Florence-2 140.0 vs BLIP 106.2 CIDEr
  Detection:  Florence-2 39.9 vs DETR 30.5 mAP (both without confidence scores)


## Step 9 · Save the clean results
Adds the clean BLIP and DETR results to `experiments/results/results.csv`, in the same format as the Florence-2 rows: 2 BLIP rows (CIDEr, BLEU-4) and 10 DETR rows (5 metrics for each protocol). The matched rows record the threshold chosen in Step 7. If the rows are already in the file, nothing is added, so rerunning the notebook never creates duplicates.

**Result:** the file shows the Florence-2 rows followed by the 12 new rows.

In [9]:
import datetime
from src.evaluate import save_result

today = datetime.date.today().isoformat()

# Do not add rows twice if the notebook is rerun
already = results[results.model.isin(["BLIP-base", "DETR-ResNet-50"]) & (results.blur_px == 0)]
if len(already):
  print(f"Clean BLIP/DETR rows are already saved ({len(already)} rows); nothing added.")
else:
  # BLIP: one row per metric
  for metric in ("CIDEr", "BLEU-4"):
    save_result({"model": "BLIP-base", "task": "captioning", "blur_type": "none", "blur_px": 0, "seed": 0,
                 "metric": metric, "score": round(blip[metric], 4), "n_images": blip["n_images"], "date": today,
                 "notes": f"clean baseline; 3-beam caption generation; no prompt; {blip['dtype']}"})
  # DETR: one row per metric and protocol
  for protocol, scores, how in [("official", detr["official"], "DETR confidence scores"),
                                ("matched", detr_matched, f"score >= {BEST_T} kept; constant score 1.0")]:
    for metric in ("mAP", "mAP50", "AP_small", "AP_medium", "AP_large"):
      save_result({"model": "DETR-ResNet-50", "task": "detection", "blur_type": "none", "blur_px": 0, "seed": 0,
                   "metric": metric, "score": round(scores[metric], 4), "n_images": detr["n_images"],
                   "date": today, "notes": f"clean baseline; {protocol} protocol: {how}"})

# Show the whole file
!cat experiments/results/results.csv

model,task,blur_type,blur_px,seed,metric,score,n_images,date,notes
Florence-2-base-ft,captioning,none,0,0,CIDEr,140.0123,5000,2026-11-18,clean baseline; 3-beam caption generation
Florence-2-base-ft,captioning,none,0,0,BLEU-4,41.424,5000,2026-11-18,clean baseline; 3-beam caption generation
Florence-2-base-ft,detection,none,0,0,mAP,39.9452,5000,2026-11-18,clean baseline; constant score 1.0; 9 fixed category aliases
Florence-2-base-ft,detection,none,0,0,mAP50,56.3713,5000,2026-11-18,clean baseline; constant score 1.0; 9 fixed category aliases
Florence-2-base-ft,detection,none,0,0,AP_small,22.4386,5000,2026-11-18,clean baseline; constant score 1.0; 9 fixed category aliases
Florence-2-base-ft,detection,none,0,0,AP_medium,44.3374,5000,2026-11-18,clean baseline; constant score 1.0; 9 fixed category aliases
Florence-2-base-ft,detection,none,0,0,AP_large,58.5103,5000,2026-11-18,clean baseline; constant score 1.0; 9 fixed category aliases
BLIP-base,captioning,none,0,0,CIDEr,106.1629,5000,2026-11

## Summary
- **BLIP-base (captioning):** the best setting is no prompt, float16, 3 beams, up to 128 new tokens. The paper's "a picture of" prompt lowered CIDEr by about 13 points and produced garbled captions; precision and the length limit made no difference. Clean score: **106.2 CIDEr**, about 27 below the published 133.3.
- **DETR-R50 (detection):** official **42.07 mAP**, matching the published 42.0. Matched protocol at the best threshold (0.925): **30.5 mAP**, up from 26.3 at 0.5.
- **Replication:** Florence-2 (140.0 CIDEr) and DETR (42.1 mAP) reproduce their published scores, so the pipeline is sound; BLIP's gap comes from its Hugging Face version.
- **Clean starting points for the claim:** captioning, Florence-2 140.0 vs BLIP 106.2 CIDEr; detection, Florence-2 39.9 vs DETR 30.5 mAP, both scored without confidence scores. Florence-2 starts ahead on both tasks.
- **Results file:** 12 clean rows added to `experiments/results/results.csv`.